# HomeGuard Security System Simulator

*Author*: Dilia Navarro

*Description*: A smart home monitoring system that processes sensor readings and triggers alerts for security, safety, and comfort issues.
             

In [5]:
import random  #random for generating sensor readings
from datetime import datetime #datetime for timestamps
 
# System configuration
HOME_MODES = ["HOME", "AWAY", "SLEEP"]
ALERT_SEVERITIES = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
 
# Current system state
current_mode = "AWAY"

Step2: Creating Data Structures

In [26]:
def create_sensor(sensor_id, location, sensor_type, threshold=None):
    sensor = {                 #library for creating a sensor with given parameters
        "id": sensor_id,
        "location": location,
        "type": sensor_type,
        "threshold": threshold
    }
    
    return sensor

def create_alert(severity, message, sensor_id, timestamp):
    alert = {                  #library for creating an alert with given parameters
        "severity": severity,
        "message": message,
        "sensor_id": sensor_id,
        "timestamp": timestamp
    }
    
    return alert

sensors = [
    create_sensor(sensor_id=1, location="Living Room", sensor_type="Motion"), #motion for the living room 
    create_sensor(sensor_id=2, location="Kitchen", sensor_type="Temperature", threshold=35), #temperature for the kitchen with threshold
    create_sensor(sensor_id=3, location="Front Door", sensor_type="Door"), #door for the front door
    create_sensor(sensor_id=4, location="Bedroom", sensor_type="Smoke") #smoke for the bedroom
]

In [22]:
print(f"Initialized {len(sensors)} sensors")
for sensor in sensors:
    print(f"  - {sensor['id']}: {sensor['location']} ({sensor['type']})")

Initialized 4 sensors
  - 1: Living Room (Motion)
  - 2: Kitchen (Temperature)
  - 3: Front Door (Door)
  - 4: Bedroom (Smoke)


Step3: Implementing If/Else Logic

In [51]:
def is_abnormal_reading(sensor, reading_value):
    if sensor["type"] == "Temperature" and reading_value > 35 or reading_value < 1: #temperature is abnormal if it's above 35 or below 1 degree Celsius
        return True
    elif sensor["type"] == "Motion" and reading_value == True: #check if motion is detected
        return True
    elif sensor["type"] == "Door" and reading_value == "OPEN": #check if the door is open
        return True
    elif sensor["type"] == "Smoke" and reading_value == "DETECTED": #check if smoke is detected
        return True
    else:
        return False
    
    
def should_trigger_security_alert(sensor, reading_value, system_mode):
    if system_mode == "AWAY":
        if sensor["type"] == "Motion" and reading_value == True: #trigger security alert if motion is detected while in AWAY mode
            return True
        elif sensor["type"] == "Door" and reading_value == "OPEN": #trigger security alert if the door is open while in AWAY mode
            return True
        elif sensor["type"] == "Smoke" and reading_value == "DETECTED": #trigger security alert if smoke is detected while in AWAY mode
            return True
        elif sensor["type"] == "Temperature" and (reading_value > 35 or reading_value < 1): #trigger security alert if temperature is abnormal while in AWAY mode
            return True
    return False

In [46]:
# Test temperature check
test_sensor = create_sensor("TEMP_TEST", "Test Room", "Temperature", threshold=1)
print(f"0°C is abnormal: {is_abnormal_reading(test_sensor, 0)}")  # Should be True
print(f"25°C is abnormal: {is_abnormal_reading(test_sensor, 25)}")  # Should be False
 
# Test security alert
motion_sensor = create_sensor("MOTION_TEST", "Test Room", "Motion")
print(f"Motion in AWAY mode triggers alert: {should_trigger_security_alert(motion_sensor, True, 'AWAY')}")  # Should be True
print(f"Motion in HOME mode triggers alert: {should_trigger_security_alert(motion_sensor, True, 'HOME')}")  # Should be False

0°C is abnormal: True
25°C is abnormal: False
Motion in AWAY mode triggers alert: True
Motion in HOME mode triggers alert: False


In [50]:
# Test temperature check
test_sensor = create_sensor("TEMP_TEST", "Test Room", "Temperature", threshold=35)
print(f"5°C is abnormal: {is_abnormal_reading(test_sensor, 5)}")  # Should be False
print(f"45°C is abnormal: {is_abnormal_reading(test_sensor, 45)}")  # Should be True

5°C is abnormal: False
45°C is abnormal: True


Step 4: Building Functions

In [83]:
def generate_reading(sensor):
    if sensor["type"] == "Temperature":
        return round(random.uniform(-1, 38),1)  # Simulate temperature between -1 and 38°C
    elif sensor["type"] == "Motion":
        return random.choice([True, False])  # Simulate motion detected or not
    elif sensor["type"] == "Door":
        return random.choice(["OPEN", "CLOSED"])  # Simulate door open or closed    
    elif sensor["type"] == "Smoke":
        return random.choice(["DETECTED", "CLEAR"])  # Simulate smoke detected or clear
    else:
        return None
  
   
def process_reading(sensor, reading_value, system_mode):
    alerts = []
    timestamp = datetime.now().strftime("%H:%M:%S")
    if system_mode == "AWAY":
        if sensor["type"] == "Motion" and reading_value == True:
            alert = create_alert(
                severity="HIGH",
                message=f"Motion detected in {sensor['location']} while AWAY",
                sensor_id=sensor["id"],
                timestamp=timestamp
            )
            alerts.append(alert)
        elif sensor["type"] == "Door" and reading_value == "OPEN":
            alert = create_alert(
                severity="HIGH",
                message=f"Door opened in {sensor['location']} while AWAY",
                sensor_id=sensor["id"],
                timestamp=timestamp
            )
            alerts.append(alert)    
        elif sensor["type"] == "Smoke" and reading_value == "DETECTED":
            alert = create_alert(
                severity="CRITICAL",
                message=f"Smoke detected in {sensor['location']} while AWAY",
                sensor_id=sensor["id"],
                timestamp=timestamp
            )
            alerts.append(alert)
        elif sensor["type"] == "Temperature" and reading_value > 35:
            alert = create_alert(
                severity="CRITICAL",
                message=f"Abnormal temperature ({reading_value}°C) in {sensor['location']} while AWAY",
                sensor_id=sensor["id"],
                timestamp=timestamp
            )
            alerts.append(alert)
        elif sensor["type"] == "Temperature" and reading_value < 1:
            alert = create_alert(
                severity="HIGH",
                message=f"Abnormal temperature ({reading_value}°C) in {sensor['location']} while AWAY",
                sensor_id=sensor["id"],
                timestamp=timestamp
            )
            alerts.append(alert)
    if system_mode == "HOME" and sensor["type"] == "Temperature":
        if 18 <= reading_value <= 23:
            alert = create_alert(
                severity="LOW",
                message=f"Abjust temperature ({reading_value}°C) in {sensor['location']} while HOME",
                sensor_id=sensor["id"],
                timestamp=timestamp
            )
            alerts.append(alert)
            
    return alerts

def trigger_alert(alert):
    severity_symbol = {
        "LOW":      "ℹ️",
        "MEDIUM":   "⚠️",
        "HIGH":     "🚨",
        "CRITICAL": "🔥",
    }
    symbol = severity_symbol.get(alert["severity"], "❓")
    print(f"[ALERT!]{symbol} {alert['severity']}: {alert['message']}")
    
def log_event(message, timestamp=None):
    if timestamp is None:
        timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[LOG] [{timestamp}] {message}")

In [84]:
# Test reading generation
test_sensor = sensors[0]  # Motion sensor
reading = generate_reading(test_sensor)
print(f"Generated reading for {test_sensor['location']}: {reading}")

Generated reading for Living Room: False


In [75]:
# Test processing
alerts = process_reading(test_sensor, True, "AWAY")
if alerts:
    trigger_alert(alerts[0])

[ALERT!]🚨 HIGH: Motion detected in Living Room while AWAY


Step 5: Creating Classes

In [ ]:
class Sensor:
    def __init__(self, sensor_id, location, sensor_type, threshold=None):
        self.id = sensor_id
        self.location = location
        self.type = sensor_type
        self.threshold = threshold
        self.current_value = None
        
    def read(self): #generates random reading based on sensor type and updates current_value
        if self.type == "Temperature":
            self.current_value = round(random.uniform(-1, 38),1)  # Simulate temperature between -1 and 38°C
        elif self.type == "Motion":
            self.current_value = random.choice([True, False])  # Simulate motion detected or not
        elif self.type == "Door":
            self.current_value = random.choice(["OPEN", "CLOSED"])  # Simulate door open or closed    
        elif self.type == "Smoke":
            self.current_value = random.choice(["DETECTED", "CLEAR"])  # Simulate smoke detected or clear
        else:
            return self.current_value
        return self.current_value
    
    
    def isAbnormal(self):  
        """
        Rules:
        - temperature : above threshold (default 35 °C) OR below 1 °C
        - motion      : True  (motion present)
        - door        : "OPEN"
        - smoke       : "DETECTED"

        Returns:
        - True if abnormal, False otherwise
        - False if no reading has been taken yet
        """ 
        if self.current_value is None:
            return False

        if self.type == "Temperature":
            high = self.threshold if self.threshold is not None else 35
            return self.current_value >= high or self.current_value <= 1

        elif self.type == "Motion":
            return self.current_value is True

        elif self.type == "Door":
            return self.current_value == "OPEN"

        elif self.type == "Smoke":
            return self.current_value == "DETECTED"

        return False
    
    def reset(self):
        self.current_value = None
        
    def __str__(self):
        status = "No reading" if self.current_value is None else str(self.current_value)
        return f"{self.id} ({self.location}): {status}"
    
sensor_objects = [
    Sensor("MOTION_001", "Living Room", "Motion"),
    Sensor("TEMP_001", "Kitchen", "Temperature", threshold=35),
    Sensor("DOOR_001", "Front Door", "Door"),
    Sensor("SMOKE_001", "Bedroom", "Smoke")
]

In [ ]:
# Create and test a sensor
test_sensor = Sensor("TEST_001", "Test Room", "Temperature", threshold=35)
test_sensor.read()
print(f"Sensor reading: {test_sensor.current_value}")
print(f"Is abnormal: {test_sensor.isAbnormal()}")
print(f"Sensor info: {test_sensor}")

Sensor reading: None
Is abnormal: False
Sensor info: TEST_001 (Test Room): No reading


Step 6: Integrating Everything

In [85]:
def run_simulation(duration_minutes=5, system_mode="AWAY"):
    print("=" * 50)
    print("=== HomeGuard Security System ===")
    print("=" * 50)
    print(f"Mode: {system_mode}\n")
    
    # Use sensor objects instead of dictionaries
    sensors = [
        Sensor("MOTION_001", "Living Room", "Motion"),
        Sensor("TEMP_001", "Kitchen", "Temperature", threshold=35),
        Sensor("DOOR_001", "Front Door", "Door"),
        Sensor("SMOKE_001", "Bedroom", "Smoke")
    ]
    
    # Simulate time passing (each iteration = 1 minute)
    for minute in range(duration_minutes):
        current_time = datetime.now().strftime("%H:%M:%S")
        print(f"\nTime: {current_time}")
        
        # Read all sensors
        for sensor in sensors:
            reading = sensor.read()
            
            # Display the reading
            if sensor.type == "temperature":
                status = "Normal" if 18 <= reading <= 23 else "Abnormal"
                print(f"[READING] {sensor.location} Temperature: {reading}°F ({status})")
            elif sensor.type == "motion":
                status = "DETECTED" if reading else "No activity"
                print(f"[READING] {sensor.location} Motion: {status}")
            elif sensor.type == "door":
                print(f"[READING] {sensor.location}: {reading}")
            elif sensor.type == "smoke":
                print(f"[READING] {sensor.location} Smoke: {reading}")
            
            # Process the reading and trigger alerts if needed
            # Convert sensor object to dict format for process_reading function
            sensor_dict = {
                "id": sensor.id,
                "location": sensor.location,
                "type": sensor.type,
                "threshold": sensor.threshold
            }
            alerts = process_reading(sensor_dict, reading, system_mode)
            
            # Trigger all alerts
            for alert in alerts:
                trigger_alert(alert)
                if alert["severity"] in ["HIGH", "CRITICAL"]:
                    log_event("Sending notification to homeowner...")
        
        # Small delay to make output readable (optional)
        import time
        time.sleep(0.5)  # 0.5 second delay between iterations
    
    print("\n" + "=" * 50)
    print("Simulation complete!")
    print("=" * 50)
 
# Main execution
if __name__ == "__main__":
    # Run the simulation
    run_simulation(duration_minutes=3, system_mode="AWAY")
    

=== HomeGuard Security System ===
Mode: AWAY


Time: 19:08:29
[ALERT!]🚨 HIGH: Motion detected in Living Room while AWAY
[LOG] [19:08:29] Sending notification to homeowner...

Time: 19:08:29
[ALERT!]🚨 HIGH: Motion detected in Living Room while AWAY
[LOG] [19:08:29] Sending notification to homeowner...

Time: 19:08:30
[ALERT!]🚨 HIGH: Door opened in Front Door while AWAY
[LOG] [19:08:30] Sending notification to homeowner...
[ALERT!]🔥 CRITICAL: Smoke detected in Bedroom while AWAY
[LOG] [19:08:30] Sending notification to homeowner...

Simulation complete!
